In [ ]:
!pip install sentence-transformers torch

In [ ]:
from datasets import load_from_disk, load_dataset

from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
dataset = load_dataset('sentence-transformers/all-nli', "triplet")

README.md: 0.00B [00:00, ?B/s]

triplet/train-00000-of-00001.parquet:   0%|          | 0.00/38.4M [00:00<?, ?B/s]

triplet/dev-00000-of-00001.parquet:   0%|          | 0.00/782k [00:00<?, ?B/s]

triplet/test-00000-of-00001.parquet:   0%|          | 0.00/810k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/557850 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/6584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6609 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 557850
    })
    dev: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 6584
    })
    test: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 6609
    })
})

In [ ]:
dataset['train'][0]

{'anchor': 'A person on a horse jumps over a broken down airplane.',
 'positive': 'A person is outdoors, on a horse.',
 'negative': 'A person is at a diner, ordering an omelette.'}

In [ ]:
dataset_train = dataset['train'].shuffle()

In [ ]:
dataset_train[0]

{'anchor': 'Two people standing in front of a yellow building with a painted mural of an athlete, a rock group, and graffiti.',
 'positive': 'Two people stand outside.',
 'negative': 'Two people sit on a bench watching people walk by'}

In [ ]:
train_dataset = dataset_train.select(range(10000))

In [ ]:
train_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 10000
})

In [ ]:
eval_dataset = dataset['dev']

In [ ]:
eval_dataset

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 6584
})

In [ ]:
loss = MultipleNegativesRankingLoss(model)

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir="models/finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy='epoch'
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.train()

Step,Training Loss
10,0.160051
20,0.265358
30,0.368479
40,0.333310
50,0.128483
60,0.176641
70,0.087369
80,0.179772
90,0.126201
100,0.137183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.13040130368669828, metrics={'train_runtime': 679.378, 'train_samples_per_second': 44.158, 'train_steps_per_second': 5.52, 'total_flos': 0.0, 'train_loss': 0.13040130368669828, 'epoch': 3.0})

In [ ]:
trainer.save_model("models/finetuned_model")
model.save("models/finetuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("models/finetuned_model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
sentences = [
    "A man is riding a horse",
    "A person is riding an animal",
    "A woman is cooking pasta"
]

embeddings = model.encode(sentences)

from sklearn.metrics.pairwise import cosine_similarity
print(cosine_similarity([embeddings[0]], embeddings))

[[ 1.          0.44931826 -0.01575969]]


In [ ]:
from sentence_transformers.evaluation import TripletEvaluator

In [ ]:
anchors = eval_dataset['anchor']
positives = eval_dataset['positive']
negatives = eval_dataset['negative']

In [ ]:
evaluator = TripletEvaluator(
    anchors=anchors,
    positives=positives,
    negatives=negatives,
    name="nli-eval"
)

In [ ]:
score = evaluator(model)

print("Triplet Accuracy:", score)

Triplet Accuracy: {'nli-eval_cosine_accuracy': 0.9595990180969238}


In [ ]:
base_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

base_score = evaluator(base_model)

print("Base model:", base_score)
print("Fine tuned:", score)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model: {'nli-eval_cosine_accuracy': 0.9249696135520935}
Fine tuned: {'nli-eval_cosine_accuracy': 0.9595990180969238}
